In [2]:

import os
import time
import warnings
from collections import Counter

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")

# ============================================================
# Problem 3 — Crypto Blind Anomaly Hunt (updated, clean output)
# ------------------------------------------------------------
# Goal:
#   Build submission.csv with:
#   symbol, date, trade_id, violation_type, remarks
#
# Design notes:
#   1) Use market files as the baseline and score trades against them.
#   2) Use manager-linked trade groups as the highest-confidence campaigns.
#   3) Keep violation_type values inside the exact accepted taxonomy.
#   4) Print clear runtime and summary logs for screenshots.
# ============================================================

DATA_DIR = "data_dir"
MARKET_DIR = os.path.join(DATA_DIR, "crypto-market")
TRADES_DIR = os.path.join(DATA_DIR, "crypto-trades")

PAIRS = [
    "USDCUSDT", "BATUSDT", "DOGEUSDT", "LTCUSDT",
    "XRPUSDT", "SOLUSDT", "ETHUSDT", "BTCUSDT",
]

ACCEPTED_VIOLATION_TYPES = {
    "aml_structuring",
    "coordinated_structuring",
    "threshold_testing",
    "chain_layering",
    "manager_consolidation",
    "placement_smurfing",
    "wash_trading",
    "pump_and_dump",
    "layering_echo",
    "spoofing",
    "ramping",
    "coordinated_pump",
    "round_trip_wash",
    "cross_pair_divergence",
    "peg_break",
    "wash_volume_at_peg",
}

ROUND_THRESHOLDS = np.array([10, 50, 100, 200, 500, 1000, 5000, 10000], dtype=float)


def rolling_z(series: pd.Series, window: int = 240, min_periods: int = 30) -> pd.Series:
    mean_ = series.rolling(window, min_periods=min_periods).mean()
    std_ = series.rolling(window, min_periods=min_periods).std().replace(0, np.nan)
    return (series - mean_) / std_


def nearest_threshold(value: float) -> tuple[float, float]:
    if pd.isna(value) or value <= 0:
        return np.nan, np.nan
    threshold = float(ROUND_THRESHOLDS[np.argmin(np.abs(ROUND_THRESHOLDS - value))])
    return threshold, float(value / threshold)


def safe_cv(series: pd.Series) -> float:
    mean_val = float(series.mean()) if len(series) else np.nan
    if pd.isna(mean_val) or mean_val == 0:
        return np.inf
    std_val = float(series.std(ddof=1)) if len(series) > 1 else 0.0
    return std_val / mean_val


def load_pair(symbol: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    market_path = os.path.join(MARKET_DIR, f"Binance_{symbol}_2026_minute.csv")
    trades_path = os.path.join(TRADES_DIR, f"{symbol}_trades.csv")

    market = pd.read_csv(market_path)
    market = market.rename(columns={"Date": "bar_time", "Open": "open", "High": "high", "Low": "low", "Close": "close"})
    market["bar_time"] = pd.to_datetime(market["bar_time"])

    asset_volume_col = [c for c in market.columns if c.startswith("Volume ") and "USDT" not in c][0]
    market = market.rename(columns={asset_volume_col: "volume_asset", "Volume USDT": "volume_usdt"})
    market = market.sort_values("bar_time").reset_index(drop=True)

    market["ret_1m"] = market["close"].pct_change()
    market["ret_fwd_5m"] = market["close"].shift(-5) / market["close"] - 1
    market["ret_fwd_15m"] = market["close"].shift(-15) / market["close"] - 1
    market["volume_z"] = rolling_z(market["volume_usdt"].fillna(0))
    market["tradecount_z"] = rolling_z(market["tradecount"].fillna(0))

    trades = pd.read_csv(trades_path, parse_dates=["timestamp"])
    trades = trades.sort_values("timestamp").reset_index(drop=True)
    trades["symbol"] = symbol
    trades["date"] = trades["timestamp"].dt.strftime("%Y-%m-%d")
    trades["minute"] = trades["timestamp"].dt.floor("min")
    trades["notional"] = trades["price"] * trades["quantity"]

    trades["wallet_first_seen"] = trades.groupby("trader_id")["timestamp"].transform("min")
    trades["is_first_day_for_wallet"] = trades["wallet_first_seen"].dt.date == trades["timestamp"].dt.date

    trades = trades.merge(
        market[
            ["bar_time", "close", "high", "low", "volume_usdt", "tradecount", "volume_z", "tradecount_z", "ret_fwd_5m", "ret_fwd_15m"]
        ],
        left_on="minute",
        right_on="bar_time",
        how="left",
    )
    trades["price_dev_bps"] = (trades["price"] / trades["close"] - 1) * 10000
    return market, trades


def event_features(df: pd.DataFrame) -> dict:
    df = df.sort_values("timestamp").copy()

    buy_notional = df.loc[df["side"] == "BUY", "notional"].sum()
    sell_notional = df.loc[df["side"] == "SELL", "notional"].sum()
    total_notional = max(buy_notional + sell_notional, 1e-9)

    prices = df["price"].to_numpy()
    if len(prices) >= 2:
        diffs = np.diff(prices)
        up_ratio = float((diffs > 0).mean())
        down_ratio = float((diffs < 0).mean())
    else:
        up_ratio = 0.0
        down_ratio = 0.0

    threshold_pairs = [nearest_threshold(x) for x in df["notional"]]
    threshold_ratios = np.array([r for _, r in threshold_pairs], dtype=float)

    return {
        "n": len(df),
        "wallets": int(df["trader_id"].nunique()),
        "buy_ratio": float((df["side"] == "BUY").mean()),
        "sell_ratio": float((df["side"] == "SELL").mean()),
        "net_notional_ratio": float(abs(buy_notional - sell_notional) / total_notional),
        "price_cv": safe_cv(df["price"]),
        "qty_cv": safe_cv(df["quantity"]),
        "notional_cv": safe_cv(df["notional"]),
        "up_ratio": up_ratio,
        "down_ratio": down_ratio,
        "event_return": float(df["price"].iloc[-1] / df["price"].iloc[0] - 1) if len(df) > 1 and df["price"].iloc[0] != 0 else 0.0,
        "future_ret_5m": float(df["ret_fwd_5m"].median()) if df["ret_fwd_5m"].notna().any() else np.nan,
        "future_ret_15m": float(df["ret_fwd_15m"].median()) if df["ret_fwd_15m"].notna().any() else np.nan,
        "volume_z_mean": float(df["volume_z"].replace([np.inf, -np.inf], np.nan).mean()) if "volume_z" in df else np.nan,
        "tradecount_z_mean": float(df["tradecount_z"].replace([np.inf, -np.inf], np.nan).mean()) if "tradecount_z" in df else np.nan,
        "near_threshold_ratio": float(((threshold_ratios >= 0.97) & (threshold_ratios < 1.00)).mean()) if len(df) else 0.0,
        "below_threshold_ratio": float((threshold_ratios < 1.00).mean()) if len(df) else 0.0,
        "first_day_wallets": int(df.groupby("trader_id")["is_first_day_for_wallet"].first().sum()),
        "median_notional": float(df["notional"].median()) if len(df) else np.nan,
        "alternating_side_ratio": float((df["side"].shift() != df["side"]).iloc[1:].mean()) if len(df) >= 2 else 0.0,
    }


def classify_event(symbol: str, event_df: pd.DataFrame) -> str:
    event_df = event_df.sort_values("timestamp").copy()
    feats = event_features(event_df)

    # Stablecoin rules are highest confidence and directly grounded in the instructions.
    if symbol == "USDCUSDT":
        if (event_df["price"].sub(1.0).abs() > 0.005).any():
            return "peg_break"
        return "wash_volume_at_peg"

    # Explicit manager consolidation rows.
    if len(event_df) == 1:
        only_row = event_df.iloc[0]
        if str(only_row["trader_id"]).startswith("mgr_") or str(only_row["trader_id"]) == str(only_row["manager_id"]):
            return "manager_consolidation"

    # Threshold testing: first trade hits the line, later trades sit just below it.
    for _, wallet_df in event_df.groupby("trader_id"):
        wallet_df = wallet_df.sort_values("timestamp")
        if len(wallet_df) >= 3:
            first_threshold, first_ratio = nearest_threshold(wallet_df["notional"].iloc[0])
            later_ratios = np.array([nearest_threshold(x)[1] for x in wallet_df["notional"].iloc[1:]], dtype=float)
            if pd.notna(first_threshold) and 0.99 <= first_ratio <= 1.01 and len(later_ratios) >= 2:
                if np.all((later_ratios >= 0.88) & (later_ratios < 0.995)):
                    return "threshold_testing"

    # Tight alternating buy/sell loops with low net change.
    if feats["n"] >= 3 and feats["net_notional_ratio"] < 0.12 and feats["price_cv"] < 0.01:
        if feats["wallets"] <= 2:
            if feats["alternating_side_ratio"] >= 0.60:
                return "round_trip_wash"
            return "wash_trading"
        return "wash_trading"

    # Structuring families.
    if feats["wallets"] >= 2 and feats["n"] >= 4:
        if feats["buy_ratio"] >= 0.70 and (feats["near_threshold_ratio"] >= 0.40 or (feats["below_threshold_ratio"] >= 0.80 and feats["notional_cv"] < 0.15)):
            if feats["first_day_wallets"] >= 2 and feats["median_notional"] < 50:
                return "placement_smurfing"
            return "coordinated_structuring"

    if feats["wallets"] == 1 and feats["n"] >= 4:
        if feats["buy_ratio"] >= 0.70 and (feats["near_threshold_ratio"] >= 0.30 or feats["notional_cv"] < 0.15):
            return "aml_structuring"

    # Multi-wallet stepped sequences.
    if feats["wallets"] >= 2 and feats["n"] >= 3 and feats["alternating_side_ratio"] >= 0.70 and feats["up_ratio"] >= 0.60:
        return "chain_layering"

    # One-wallet push then reverse.
    for _, wallet_df in event_df.groupby("trader_id"):
        wallet_df = wallet_df.sort_values("timestamp")
        if len(wallet_df) >= 4:
            sides = wallet_df["side"].tolist()
            if (sides.count("BUY") >= 3 and sides[-1] == "SELL") or (sides.count("SELL") >= 3 and sides[-1] == "BUY"):
                if wallet_df["notional"].iloc[-1] >= 1.8 * wallet_df["notional"].median():
                    return "layering_echo"

    # Pump/ramping families guided by price path and market context.
    if feats["buy_ratio"] >= 0.75 and feats["up_ratio"] >= 0.60 and feats["event_return"] > 0:
        future_reversal = (
            pd.notna(feats["future_ret_15m"]) and feats["future_ret_15m"] < -max(0.002, 0.5 * abs(feats["event_return"]))
        )
        market_confirmation = (
            (pd.notna(feats["volume_z_mean"]) and feats["volume_z_mean"] > 1.0)
            or (pd.notna(feats["tradecount_z_mean"]) and feats["tradecount_z_mean"] > 1.0)
        )
        if future_reversal and market_confirmation:
            if feats["wallets"] >= 2:
                return "coordinated_pump"
            return "pump_and_dump"
        if feats["wallets"] >= 2:
            return "coordinated_pump"
        return "ramping"

    # Conservative fallbacks.
    if feats["wallets"] >= 2 and feats["buy_ratio"] >= 0.70:
        return "coordinated_pump"
    if feats["wallets"] == 1 and feats["buy_ratio"] >= 0.70:
        return "ramping"
    if feats["wallets"] >= 2:
        return "coordinated_structuring"
    return "aml_structuring"


def make_remark(row: pd.Series, event_df: pd.DataFrame, violation_type: str) -> str:
    event_df = event_df.sort_values("timestamp")
    feats = event_features(event_df)
    wallet = row["trader_id"]
    manager = row["manager_id"]
    date = row["date"]

    if violation_type == "peg_break":
        deviation_pct = abs(row["price"] - 1.0) * 100
        return (
            f"USDC trade moved {deviation_pct:.3f}% away from the $1.00 peg at {row['price']:.6f}; "
            f"group={manager}, wallet={wallet}, qty={row['quantity']:.4f}."
        )

    if violation_type == "wash_volume_at_peg":
        return (
            f"Linked USDC activity stayed pinned near $1.00 with repeated offsetting flow; "
            f"group={manager}, n={feats['n']}, net_notional_ratio={feats['net_notional_ratio']:.3f}."
        )

    if violation_type == "manager_consolidation":
        return (
            f"Late large consolidation-style transfer inside linked group {manager}; "
            f"wallet={wallet}, notional={row['notional']:.2f}, date={date}."
        )

    if violation_type == "threshold_testing":
        threshold, ratio = nearest_threshold(row["notional"])
        return (
            f"Wallet={wallet} touched a round-number threshold and follow-up trades stayed just under it; "
            f"current notional={row['notional']:.2f}, nearest_threshold={threshold:.0f}, group={manager}."
        )

    if violation_type == "aml_structuring":
        return (
            f"Single-wallet repeated similar-size flow on {date}; wallet={wallet}, n={feats['n']}, "
            f"notional_cv={feats['notional_cv']:.3f}, buy_ratio={feats['buy_ratio']:.2f}, group={manager}."
        )

    if violation_type == "coordinated_structuring":
        return (
            f"Multi-wallet below-threshold pattern on {date}; wallets={feats['wallets']}, n={feats['n']}, "
            f"notional_cv={feats['notional_cv']:.3f}, group={manager}."
        )

    if violation_type == "placement_smurfing":
        return (
            f"Multiple first-day wallets placed coordinated small buys; first_day_wallets={feats['first_day_wallets']}, "
            f"median_notional={feats['median_notional']:.2f}, group={manager}."
        )

    if violation_type == "round_trip_wash":
        return (
            f"Tight alternating buy/sell loop with low net exposure change; net_notional_ratio={feats['net_notional_ratio']:.3f}, "
            f"wallets={feats['wallets']}, group={manager}."
        )

    if violation_type == "wash_trading":
        return (
            f"Linked activity shows near-flat net positioning and a tight price band; "
            f"net_notional_ratio={feats['net_notional_ratio']:.3f}, price_cv={feats['price_cv']:.4f}, group={manager}."
        )

    if violation_type == "chain_layering":
        return (
            f"Stepped multi-wallet sequence with alternating sides and upward price path; "
            f"alternating_ratio={feats['alternating_side_ratio']:.2f}, wallets={feats['wallets']}, group={manager}."
        )

    if violation_type == "layering_echo":
        return (
            f"One wallet pushed price in one direction and then reversed with a larger trade; "
            f"wallet={wallet}, event_return={feats['event_return']*100:.2f}%, group={manager}."
        )

    if violation_type == "pump_and_dump":
        return (
            f"Single-wallet buying sequence pushed price up, then forward bars reversed; "
            f"event_return={feats['event_return']*100:.2f}%, future_ret_15m={feats['future_ret_15m']*100:.2f}%, group={manager}."
        )

    if violation_type == "coordinated_pump":
        return (
            f"Linked wallets bought in the same direction while price climbed; wallets={feats['wallets']}, "
            f"event_return={feats['event_return']*100:.2f}%, avg_tradecount_z={feats['tradecount_z_mean']:.2f}, group={manager}."
        )

    if violation_type == "ramping":
        return (
            f"One wallet kept trading with an upward drift in price; wallet={wallet}, "
            f"up_ratio={feats['up_ratio']:.2f}, event_return={feats['event_return']*100:.2f}%, group={manager}."
        )

    return (
        f"Linked campaign row inside group={manager}; wallet={wallet}, side={row['side']}, "
        f"notional={row['notional']:.2f}."
    )


def classify_rows_for_symbol(symbol: str, trades: pd.DataFrame) -> list[dict]:
    linked = trades[trades["manager_id"].notna()].copy()
    if linked.empty:
        return []

    output_rows = []

    for (manager_id, date), event_df in linked.groupby(["manager_id", "date"]):
        event_df = event_df.sort_values("timestamp").copy()
        event_type = classify_event(symbol, event_df)

        # Row-level overrides to keep a few exact patterns tighter.
        if event_type in {"coordinated_structuring", "aml_structuring", "placement_smurfing"}:
            late_large_cutoff = max(event_df["notional"].median() * 3.0, event_df["notional"].quantile(0.90))
        else:
            late_large_cutoff = np.inf

        for _, row in event_df.iterrows():
            violation_type = event_type

            # Exact USDC peg labeling per row.
            if symbol == "USDCUSDT":
                violation_type = "peg_break" if abs(row["price"] - 1.0) > 0.005 else "wash_volume_at_peg"

            # Consolidation-like late large transfer.
            elif row["notional"] >= late_large_cutoff and row["timestamp"] == event_df["timestamp"].max():
                if event_type in {"coordinated_structuring", "aml_structuring", "placement_smurfing"}:
                    violation_type = "manager_consolidation"

            # Explicit manager wallet appearance.
            if str(row["trader_id"]).startswith("mgr_") or str(row["trader_id"]) == str(row["manager_id"]):
                violation_type = "manager_consolidation"

            if violation_type not in ACCEPTED_VIOLATION_TYPES:
                violation_type = "aml_structuring"

            output_rows.append(
                {
                    "symbol": symbol,
                    "date": row["date"],
                    "trade_id": row["trade_id"],
                    "violation_type": violation_type,
                    "remarks": make_remark(row, event_df, violation_type),
                }
            )

    return output_rows


def build_submission() -> pd.DataFrame:
    t0 = time.time()
    all_rows = []
    symbol_summaries = []
    type_counter = Counter()

    print("=" * 78)
    print("Problem 3 — Updated Submission Builder")
    print("=" * 78)

    for symbol in PAIRS:
        pair_start = time.time()
        market, trades = load_pair(symbol)
        linked = trades[trades["manager_id"].notna()].copy()

        print(f"\n[{symbol}]")
        print(f"  market rows={len(market):,} | trade rows={len(trades):,} | linked trades={len(linked):,} | linked groups={linked['manager_id'].dropna().nunique()}")

        symbol_rows = classify_rows_for_symbol(symbol, trades)
        symbol_df = pd.DataFrame(symbol_rows)

        if not symbol_df.empty:
            counts = symbol_df["violation_type"].value_counts().to_dict()
            for k, v in counts.items():
                type_counter[k] += v
            print(f"  flagged rows={len(symbol_df):,} | types={counts}")
            all_rows.extend(symbol_rows)
        else:
            print("  flagged rows=0")

        symbol_summaries.append(
            {
                "symbol": symbol,
                "market_rows": len(market),
                "trade_rows": len(trades),
                "linked_rows": len(linked),
                "flagged_rows": len(symbol_df),
                "runtime_sec": round(time.time() - pair_start, 2),
            }
        )

    submission = pd.DataFrame(all_rows)
    submission = submission.drop_duplicates(subset=["trade_id"], keep="first")
    submission = submission[["symbol", "date", "trade_id", "violation_type", "remarks"]]
    submission = submission.sort_values(["symbol", "date", "trade_id"]).reset_index(drop=True)

    elapsed = time.time() - t0
    submission.to_csv("submission.csv", index=False)

    print("\n" + "=" * 78)
    print("FINAL OUTPUT SUMMARY")
    print("=" * 78)
    print(f"Total flagged trades : {len(submission)}")
    print(f"Total runtime        : {elapsed:.2f} sec")
    print(f"Output file          : submission.csv")

    print("\nPer-symbol summary:")
    display(pd.DataFrame(symbol_summaries))

    print("\nViolation type counts:")
    if len(submission):
        display(submission["violation_type"].value_counts().rename_axis("violation_type").reset_index(name="count"))
    else:
        print("No rows generated.")

    print("\nSubmission preview:")
    display(submission[["symbol", "date", "trade_id", "violation_type"]].head(20))

    print("\nRemarks preview:")
    display(submission[["trade_id", "violation_type", "remarks"]].head(10))

    return submission

submission_df = build_submission()


Problem 3 — Updated Submission Builder

[USDCUSDT]
  market rows=110,862 | trade rows=1,019 | linked trades=19 | linked groups=2
  flagged rows=19 | types={'wash_volume_at_peg': 15, 'peg_break': 3, 'manager_consolidation': 1}

[BATUSDT]
  market rows=110,862 | trade rows=535 | linked trades=35 | linked groups=2
  flagged rows=35 | types={'aml_structuring': 25, 'coordinated_pump': 10}

[DOGEUSDT]
  market rows=110,862 | trade rows=2,033 | linked trades=33 | linked groups=3
  flagged rows=33 | types={'wash_trading': 10, 'ramping': 7, 'aml_structuring': 7, 'round_trip_wash': 6, 'coordinated_pump': 3}

[LTCUSDT]
  market rows=110,862 | trade rows=1,527 | linked trades=27 | linked groups=4
  flagged rows=27 | types={'coordinated_structuring': 8, 'ramping': 7, 'chain_layering': 5, 'coordinated_pump': 4, 'aml_structuring': 3}

[XRPUSDT]
  market rows=110,862 | trade rows=2,535 | linked trades=35 | linked groups=1
  flagged rows=35 | types={'aml_structuring': 15, 'coordinated_pump': 8, 'round_

,symbol,market_rows,trade_rows,linked_rows,flagged_rows,runtime_sec
0,USDCUSDT,110862,1019,19,19,0.75
1,BATUSDT,110862,535,35,35,0.74
2,DOGEUSDT,110862,2033,33,33,0.72
3,LTCUSDT,110862,1527,27,27,0.64
4,XRPUSDT,110862,2535,35,35,0.73
5,SOLUSDT,110862,2529,29,29,0.67
6,ETHUSDT,110862,4031,31,31,0.63
7,BTCUSDT,110862,5045,45,45,0.83



Violation type counts:


,violation_type,count
0,aml_structuring,81
1,wash_trading,46
2,coordinated_pump,44
3,ramping,21
4,round_trip_wash,18
5,chain_layering,15
6,wash_volume_at_peg,15
7,coordinated_structuring,8
8,manager_consolidation,3
9,peg_break,3



Submission preview:


,symbol,date,trade_id,violation_type
0,BATUSDT,2026-01-13,BATUSDT_00000501,aml_structuring
1,BATUSDT,2026-01-13,BATUSDT_00000502,aml_structuring
2,BATUSDT,2026-01-21,BATUSDT_00000503,aml_structuring
3,BATUSDT,2026-01-21,BATUSDT_00000504,aml_structuring
4,BATUSDT,2026-01-21,BATUSDT_00000505,aml_structuring
5,BATUSDT,2026-01-21,BATUSDT_00000506,aml_structuring
6,BATUSDT,2026-01-21,BATUSDT_00000507,aml_structuring
7,BATUSDT,2026-01-21,BATUSDT_00000508,aml_structuring
8,BATUSDT,2026-01-21,BATUSDT_00000509,aml_structuring
9,BATUSDT,2026-01-28,BATUSDT_00000510,aml_structuring



Remarks preview:


,trade_id,violation_type,remarks
0,BATUSDT_00000501,aml_structuring,Single-wallet repeated similar-size flow on 20...
1,BATUSDT_00000502,aml_structuring,Single-wallet repeated similar-size flow on 20...
2,BATUSDT_00000503,aml_structuring,Single-wallet repeated similar-size flow on 20...
3,BATUSDT_00000504,aml_structuring,Single-wallet repeated similar-size flow on 20...
4,BATUSDT_00000505,aml_structuring,Single-wallet repeated similar-size flow on 20...
5,BATUSDT_00000506,aml_structuring,Single-wallet repeated similar-size flow on 20...
6,BATUSDT_00000507,aml_structuring,Single-wallet repeated similar-size flow on 20...
7,BATUSDT_00000508,aml_structuring,Single-wallet repeated similar-size flow on 20...
8,BATUSDT_00000509,aml_structuring,Single-wallet repeated similar-size flow on 20...
9,BATUSDT_00000510,aml_structuring,Single-wallet repeated similar-size flow on 20...
